# Phase-6 GPU walk-forward models — Google Colab (free T4)

Trains the remaining published-GPU CN models + the US TCN on the
walk-forward grid, writing qlib `results/oof/{cn,us}/<model>__{fold,holdout}.pkl`
preds that the ensemble / cn-score jobs consume — exactly what
`MODELS='tcn_cn_wf alstm_cn_wf tra_cn_wf igmtf_cn_wf tcn_us_wf'` does on
Kaggle, for when the owner is out of Kaggle GPU quota.

## One-time setup (2 min)

1. **Runtime → Change runtime type → T4 GPU** (required — these models are `requires_gpu`).
2. **key icon (Secrets) → Add** `GH_TOKEN` = a GitHub PAT with repo read+write
   (the repo is private; the token is used for clone / artifact fetch / results push
   and is never printed).
3. Optional secret `RESULTS_BRANCH` to pin the results branch name (default
   `results/colab-wf-<today>`).

Then **Runtime → Run all**. Top-to-bottom re-runs are safe.

## Order & the 12h wall

Rough T4 times for the full 5-fold pass (4 OOF + holdout): TCN ~1-2h,
ALSTM ~1.5-2.5h, TRA ~2.5-4h, IGMTF ~3-5h, US TCN ~2-4h. All five likely
**exceed one free session** — they are ordered by expected CN upside so a
disconnect still lands the best models first:

1. `tra_cn_wf` — published IC 0.1135, novel optimal-transport attention.
2. `tcn_cn_wf` — cheap, conv architecture diversifies the GBDT/attention stack.
3. `igmtf_cn_wf` — Alpha360 MTF-graph, complements HIST but heaviest.
4. `alstm_cn_wf` — closest in family to existing models.
5. `tcn_us_wf` — US side (bonus; the CN chase is the goal).

## Disconnect resume

`phase2_run.py` already skips any `<model>__<fold>.pkl` that exists. A
background loop pushes `results/oof` to the results branch every ~7 min and
after each model; a fresh session restores those files before training, so a
disconnect loses at most the in-flight fold. Just **Run all** again — same-day
sessions hit the same dated branch automatically; next day set the
`RESULTS_BRANCH` secret to yesterday's branch.

US models need the `us-qlib-dump` workflow artifact — fetched below via
`GH_TOKEN`; no Alpaca keys in this notebook.

In [ ]:
# --- environment + secrets ---------------------------------------------------
import os, sys, subprocess, threading, time, json, glob

ON_COLAB = os.path.exists('/content')
WORK = '/content' if ON_COLAB else os.path.expanduser('~/colab-run')
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'colab' if ON_COLAB else 'local', '| workdir:', WORK)

def get_secret(name):
    """GH_TOKEN etc. — Colab userdata Secrets, falling back to env vars."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name)

GH_TOKEN = get_secret('GH_TOKEN')
if not GH_TOKEN:
    raise RuntimeError('Set GH_TOKEN in Colab Secrets (PAT with repo access)')
os.environ['GH_TOKEN'] = GH_TOKEN  # subprocesses (artifact fetch) read env


In [ ]:
# --- clone private repo @ phase6/benchmarks ----------------------------------
REPO = 'Riyu-D1/quant-research'
BRANCH = 'phase6/benchmarks'
REPO_URL = f'https://{GH_TOKEN}@github.com/{REPO}.git'
if not os.path.exists('quant-research'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '50',
                    REPO_URL, 'quant-research'], check=True)
# idempotent: re-running cells after a kernel restart must not fail on cd
if os.path.basename(os.getcwd()) != 'quant-research':
    os.chdir('quant-research')
# token lives only in .git/config — never print REPO_URL or GH_TOKEN
subprocess.run(['git', 'config', 'credential.helper', ''], check=False)
print('on branch:', subprocess.run(['git','branch','--show-current'],
      capture_output=True, text=True).stdout.strip())


In [ ]:
# --- deps (uv, py3.11, CUDA torch 2.6 via PyPI) -------------------------------
subprocess.run(['pip','-q','install','uv'], check=True)
subprocess.run(['uv','python','install','3.11'], check=True)
subprocess.run(['uv','sync','--frozen','--no-dev'], check=True)
r = subprocess.run(['.venv/bin/python','-c',
    'import torch;print(torch.__version__,"cuda:",torch.cuda.is_available())'],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if 'cuda: True' not in r.stdout:
    print('WARNING: no CUDA — Runtime -> Change runtime type -> T4 GPU. '
          'requires_gpu models will crawl on CPU.')


In [ ]:
# --- MODELS + results-branch plumbing ---------------------------------------
# Upside order (edit to trim): best CN models first so a disconnect keeps value.
import glob
MODELS = 'tra_cn_wf tcn_cn_wf igmtf_cn_wf alstm_cn_wf tcn_us_wf'.split()

RESULTS_BRANCH = (get_secret('RESULTS_BRANCH')
                  or f"results/colab-wf-{__import__('datetime').date.today()}")
# paths that hold the pred artifacts — scoped, never `git add results/`
PUSH_PATHS = ['results/oof', 'results/runs', 'results/news_sentiment']
CKPT_EVERY = 420  # seconds between background pushes

def _git(*args, check=False):
    return subprocess.run(['git', *args], capture_output=True, text=True,
                          check=check)

def restore_preds():
    """Pull prior pushed preds from the results branch into the worktree so
    phase2_run's per-fold skip sees them. No-op if the branch doesn't exist."""
    _git('fetch', 'origin', RESULTS_BRANCH)
    if _git('rev-parse', 'FETCH_HEAD').returncode != 0:
        print('no results branch yet — fresh start')
        return 0
    n = 0
    for p in PUSH_PATHS:
        if _git('restore', '--source=FETCH_HEAD', '--worktree', p).returncode == 0:
            n += 1
    got = _git('status', '--porcelain', '--', 'results/oof').stdout.count('\n')
    print(f'restored preds from origin/{RESULTS_BRANCH}: {got} path(s)')
    return got

def ckpt(tag=''):
    """Commit pred files onto HEAD and push HEAD:<results branch>.
    force-with-lease in the fallback because a second session's base can differ
    from the first push's base (results files are additive — safe)."""
    for p in PUSH_PATHS:
        if os.path.exists(p):
            _git('add', p)
    if not _git('diff', '--cached', '--quiet').returncode:
        print('ckpt: nothing new'); return
    _git('-c', 'user.name=colab-gpu', '-c',
         'user.email=noreply@users.noreply.github.com',
         'commit', '-qm', f'colab gpu preds {tag}'.strip())
    r = _git('push', '-q', 'origin', f'HEAD:{RESULTS_BRANCH}')
    if r.returncode:
        _git('fetch', 'origin', RESULTS_BRANCH)
        r = _git('push', '-q', '--force-with-lease', 'origin',
                 f'HEAD:{RESULTS_BRANCH}')
    print('ckpt pushed ->', RESULTS_BRANCH, tag, '| rc', r.returncode)


In [ ]:
# --- restore any prior preds (idempotent; safe on first run) -----------------
restore_preds()
import glob
print('preds present:', len(glob.glob('results/oof/*/*__*.pkl')))


In [ ]:
# --- fetch the US qlib dump only if a *_us model is requested ----------------
US_DUMP = 'data/qlib/us_alpaca'
if any('_us' in m for m in MODELS) \
        and not os.path.exists(f'{US_DUMP}/calendars/day.txt'):
    import io, zipfile, urllib.request
    api = f'https://api.github.com/repos/{REPO}'
    hdr = {'Authorization': f'Bearer {GH_TOKEN}',
           'Accept': 'application/vnd.github+json'}
    req = urllib.request.Request(
        f'{api}/actions/artifacts?name=us-qlib-dump&per_page=100', headers=hdr)
    arts = [a for a in json.load(urllib.request.urlopen(req))['artifacts']
            if not a['expired']]
    if not arts:
        raise RuntimeError('no us-qlib-dump artifact — run us-alpaca-data first')
    url = sorted(arts, key=lambda a: a['created_at'])[-1]['archive_download_url']
    import requests  # urllib re-sends Authorization cross-host -> 401
    r = requests.get(url, headers=hdr, timeout=600)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    os.makedirs(US_DUMP, exist_ok=True)
    z.extractall(US_DUMP)
    print('extracted us-qlib-dump ->', US_DUMP)
else:
    print('us dump present or not needed')


In [ ]:
# --- train: sequential models, background ckpt every ~7min + per-model push --
stop = threading.Event()
fails = {}

def _ckpt_loop():
    while not stop.wait(CKPT_EVERY):
        ckpt('(periodic)')

def run_model(name):
    t0 = time.time()
    print(f'=== {name} start ({time.strftime("%H:%M:%S")})', flush=True)
    rc = subprocess.run(['.venv/bin/python', 'scripts/gpu_run.py',
                         '--models', name]).returncode
    mins = (time.time() - t0) / 60
    if rc:
        fails[name] = rc
        print(f'=== {name} FAILED rc={rc} after {mins:.0f}m — continuing', flush=True)
    else:
        print(f'=== {name} done in {mins:.0f}m', flush=True)
    ckpt(name)

_th = threading.Thread(target=_ckpt_loop, daemon=True)
_th.start()
try:
    for m in MODELS:
        run_model(m)
finally:
    stop.set()
    ckpt('(final)')

done = sorted(glob.glob('results/oof/*/*__*.pkl'))
print(f'\n{len(done)} pred files on disk; failures: {fails or "none"}')

In [ ]:
# --- final status -------------------------------------------------------------
import glob, os
for f in sorted(glob.glob('results/oof/*/*__*.pkl')):
    print(f'{f}  {os.path.getsize(f)/1e6:.1f}MB')
r = _git('log', '--oneline', '-5', f'origin/{RESULTS_BRANCH}')
print('\nresults branch tip:\n' + (r.stdout or '(not pushed yet)'))
print(f'\nNext: merge {RESULTS_BRANCH} preds into a working branch, then '
      'score ensembles/holdout on CI — never merge blindly, preds are additive.')

## How to run this (owner, 5 lines)

1. Open the badge link in PR #6 → **Runtime → Change runtime type → T4 GPU**.
2. **key icon → Secrets**: `GH_TOKEN` = your repo PAT (once; stays in Colab).
3. **Runtime → Run all**. Models run best-first (tra → tcn → igmtf → alstm → tcn_us); preds push to `results/colab-wf-<date>` every ~7 min.
4. If the session dies, just run all again — completed folds resume automatically (next day: set secret `RESULTS_BRANCH` to yesterday's branch first).
5. When done, tell me the results branch name and I'll merge the preds and score the ensembles + CN holdout.